# OASIS Pilot FreeSurfer Processing Notebook

This notebook prepares a small OASIS pilot dataset and runs FreeSurfer for the two demo subjects used in the evidence workflow.

- OAS1_0001_MR1 as a control-style example
- OAS1_0003_MR1 as a dementia-style example

Outputs from this notebook are designed to feed the later evidence-informed workflow notebook.


## Stage 1. Configure paths and pilot subjects

This notebook assumes the OASIS archive has already been extracted under /home/jovyan/neurodesk/disc1.

Generated outputs will be written to:
- /home/jovyan/converted_nifti
- /home/jovyan/derivatives/freesurfer
- /home/jovyan/outputs


In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import re

RAW_BASE = Path('/home/jovyan/neurodesk/disc1')
CONVERTED_DIR = Path('/home/jovyan/converted_nifti')
FS_DIR = Path('/home/jovyan/derivatives/freesurfer')
OUTPUT_DIR = Path('/home/jovyan/outputs')

CONVERTED_DIR.mkdir(parents=True, exist_ok=True)
FS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_SUBJECTS = [
    {'subject_id': 'OAS1_0001_MR1', 'diagnosis': 'Control'},
    {'subject_id': 'OAS1_0003_MR1', 'diagnosis': 'Dementia'},
]

config_df = pd.DataFrame([
    {'item': 'raw_base', 'value': str(RAW_BASE)},
    {'item': 'raw_base_exists', 'value': RAW_BASE.exists()},
    {'item': 'converted_dir', 'value': str(CONVERTED_DIR)},
    {'item': 'freesurfer_dir', 'value': str(FS_DIR)},
    {'item': 'output_dir', 'value': str(OUTPUT_DIR)},
])

display(config_df)
display(pd.DataFrame(PILOT_SUBJECTS))


,item,value
0,raw_base,/home/jovyan/neurodesk/disc1
1,raw_base_exists,True
2,converted_dir,/home/jovyan/converted_nifti
3,freesurfer_dir,/home/jovyan/derivatives/freesurfer
4,output_dir,/home/jovyan/outputs


,subject_id,diagnosis
0,OAS1_0001_MR1,Control
1,OAS1_0003_MR1,Dementia


## Stage 2. Inspect raw OASIS content

This checks that the selected subject folders and candidate MRI files are present.


In [2]:
def subject_inventory(subject_id):
    sdir = RAW_BASE / subject_id
    rows = []
    for label, rel in [
        ('subject_dir', Path('.')),
        ('raw_dir', Path('RAW')),
        ('processed_dir', Path('PROCESSED')),
        ('fsl_seg_dir', Path('FSL_SEG')),
        ('xml_file', Path(f'{subject_id}.xml')),
        ('txt_file', Path(f'{subject_id}.txt')),
    ]:
        rows.append({'subject_id': subject_id, 'item': label, 'exists': (sdir / rel).exists(), 'path': str(sdir / rel)})
    return pd.DataFrame(rows)

inventory_df = pd.concat([subject_inventory(item['subject_id']) for item in PILOT_SUBJECTS], ignore_index=True)
display(inventory_df)


,subject_id,item,exists,path
0,OAS1_0001_MR1,subject_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1
1,OAS1_0001_MR1,raw_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW
2,OAS1_0001_MR1,processed_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/PRO...
3,OAS1_0001_MR1,fsl_seg_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/FSL...
4,OAS1_0001_MR1,xml_file,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/OAS...
5,OAS1_0001_MR1,txt_file,True,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/OAS...
6,OAS1_0003_MR1,subject_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0003_MR1
7,OAS1_0003_MR1,raw_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0003_MR1/RAW
8,OAS1_0003_MR1,processed_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0003_MR1/PRO...
9,OAS1_0003_MR1,fsl_seg_dir,True,/home/jovyan/neurodesk/disc1/OAS1_0003_MR1/FSL...


In [7]:
def find_raw_input(subject_id):
    raw_dir = RAW_BASE / subject_id / 'RAW'

    img_candidates = sorted(raw_dir.glob('*mpr-1_anon.img'))
    if img_candidates:
        return img_candidates[0]

    hdr_candidates = sorted(raw_dir.glob('*mpr-1_anon.hdr'))
    if hdr_candidates:
        return hdr_candidates[0]

    fallback_img = sorted(raw_dir.glob('*.img'))
    if fallback_img:
        return fallback_img[0]

    fallback_hdr = sorted(raw_dir.glob('*.hdr'))
    if fallback_hdr:
        return fallback_hdr[0]

    return None


input_rows = []
for item in PILOT_SUBJECTS:
    sid = item['subject_id']
    raw_input = find_raw_input(sid)
    input_rows.append({
        'subject_id': sid,
        'diagnosis': item['diagnosis'],
        'raw_input': str(raw_input) if raw_input else None,
        'input_exists': raw_input is not None and raw_input.exists(),
        'nii_output': str(CONVERTED_DIR / f'{sid}_T1w.nii.gz'),
    })

input_df = pd.DataFrame(input_rows)
display(input_df)

,subject_id,diagnosis,raw_input,input_exists,nii_output
0,OAS1_0001_MR1,Control,/home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW...,True,/home/jovyan/converted_nifti/OAS1_0001_MR1_T1w...
1,OAS1_0003_MR1,Dementia,/home/jovyan/neurodesk/disc1/OAS1_0003_MR1/RAW...,True,/home/jovyan/converted_nifti/OAS1_0003_MR1_T1w...


## Stage 3. Load FreeSurfer and convert OASIS images to NIfTI

This stage uses mri_convert from the FreeSurfer module.


In [4]:
subprocess.run(['bash', '-lc', 'ml freesurfer/8.2.0 && which mri_convert && which recon-all'], check=True)


/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.2.0_20260601/mri_convert
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.2.0_20260601/recon-all


CompletedProcess(args=['bash', '-lc', 'ml freesurfer/8.2.0 && which mri_convert && which recon-all'], returncode=0)

In [8]:
for item in PILOT_SUBJECTS:
    sid = item['subject_id']
    raw_input = find_raw_input(sid)
    out_nii = CONVERTED_DIR / f'{sid}_T1w.nii.gz'

    if raw_input is None:
        raise FileNotFoundError(f'No RAW input found for {sid}')

    cmd = f'ml freesurfer/8.2.0 && mri_convert "{raw_input}" "{out_nii}"'
    print('Running conversion for', sid)
    print('Input:', raw_input)

    result = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True)

    print('returncode =', result.returncode)
    print('STDOUT:')
    print(result.stdout[-2000:])
    print('STDERR:')
    print(result.stderr[-2000:])

    if result.returncode != 0:
        raise RuntimeError(f'mri_convert failed for {sid}')

converted_df = pd.DataFrame([
    {
        'subject_id': item['subject_id'],
        'nii_exists': (CONVERTED_DIR / f"{item['subject_id']}_T1w.nii.gz").exists(),
        'nii_path': str(CONVERTED_DIR / f"{item['subject_id']}_T1w.nii.gz")
    }
    for item in PILOT_SUBJECTS
])

display(converted_df)

Running conversion for OAS1_0001_MR1
Input: /home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mpr-1_anon.img
returncode = 0
STDOUT:
/opt/freesurfer-8.2.0/bin/mri_convert /home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mpr-1_anon.img /home/jovyan/converted_nifti/OAS1_0001_MR1_T1w.nii.gz 
reading from /home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mpr-1_anon.img...
  analyzeRead() roi_scale   0.000000000
TR=0.00, TE=0.00, TI=0.00, flip angle=0.00
i_ras = (0, 1, 0)
j_ras = (0, 0, 1)
k_ras = (-1, 0, 0)
writing to /home/jovyan/converted_nifti/OAS1_0001_MR1_T1w.nii.gz...

STDERR:
-----------------------------------------------------------------
INFO: could not find /home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mpr-1_anon.mat file for direction cosine info.
INFO: use Analyze 7.5 hdr->hist.orient value: 2, sagittal unflipped.
INFO: if not valid, please provide the information in /home/jovyan/neurodesk/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mp

,subject_id,nii_exists,nii_path
0,OAS1_0001_MR1,True,/home/jovyan/converted_nifti/OAS1_0001_MR1_T1w...
1,OAS1_0003_MR1,True,/home/jovyan/converted_nifti/OAS1_0003_MR1_T1w...


## Stage 4. Run FreeSurfer recon-all

This step is slow. Run it once, then monitor progress in the next stage.


In [ ]:
RUN_RECON = True
TARGET_SUBJECT_IDS = {'OAS1_0003_MR1'}

for item in PILOT_SUBJECTS:
    sid = item['subject_id']
    if sid not in TARGET_SUBJECT_IDS:
        print('Skipping', sid)
        continue

    in_nii = CONVERTED_DIR / f'{sid}_T1w.nii.gz'
    log_path = OUTPUT_DIR / f'{sid}_recon.log'
    cmd = (
        f'ml freesurfer/8.2.0 && '
        f'nohup recon-all -sd "{FS_DIR}" -i "{in_nii}" -s "{sid}" -all '
        f'> "{log_path}" 2>&1 &'
    )
    print('Prepared recon-all command for', sid)
    print(cmd)
    if RUN_RECON:
        result = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True)
        print('launch_returncode =', result.returncode)
        print('STDOUT:')
        print(result.stdout[-1000:])
        print('STDERR:')
        print(result.stderr[-1000:])

status_preview = pd.DataFrame([
    {
        'subject_id': sid,
        'log_path': str(OUTPUT_DIR / f'{sid}_recon.log'),
        'nii_exists': (CONVERTED_DIR / f'{sid}_T1w.nii.gz').exists(),
        'subject_dir_exists': (FS_DIR / sid).exists(),
    }
    for sid in sorted(TARGET_SUBJECT_IDS)
])
display(status_preview)

Skipping OAS1_0001_MR1
Prepared recon-all command for OAS1_0003_MR1
ml freesurfer/8.2.0 && nohup recon-all -sd "/home/jovyan/derivatives/freesurfer" -i "/home/jovyan/converted_nifti/OAS1_0003_MR1_T1w.nii.gz" -s "OAS1_0003_MR1" -all > "/home/jovyan/outputs/OAS1_0003_MR1_recon.log" 2>&1 &


In [1]:
def freesurfer_status(subject_id):
    subject_dir = FS_DIR / subject_id
    scripts_dir = subject_dir / 'scripts'
    stats_dir = subject_dir / 'stats'
    return {
        'subject_id': subject_id,
        'subject_dir_exists': subject_dir.exists(),
        'stats_dir_exists': stats_dir.exists(),
        'aseg_exists': (stats_dir / 'aseg.stats').exists(),
        'lh_aparc_exists': (stats_dir / 'lh.aparc.stats').exists(),
        'rh_aparc_exists': (stats_dir / 'rh.aparc.stats').exists(),
        'done_exists': (scripts_dir / 'recon-all.done').exists(),
        'log_path': str(OUTPUT_DIR / f'{subject_id}_recon.log'),
    }

display(pd.DataFrame([freesurfer_status(item['subject_id']) for item in PILOT_SUBJECTS]))


NameError: name 'pd' is not defined

## Stage 5. Parse FreeSurfer stats into a structural feature CSV

This exports a small vertical table that can be passed into the evidence-informed notebook later.


In [ ]:
def parse_aseg_stats(subject_id, diagnosis):
    aseg_path = FS_DIR / subject_id / 'stats' / 'aseg.stats'
    if not aseg_path.exists():
        return []
    rows = []
    for line in aseg_path.read_text(errors='ignore').splitlines():
        if line.startswith('# Measure'):
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 4:
                roi_name = parts[1].replace('Measure ', '').strip()
                value_text = parts[3]
                try:
                    value_numeric = float(value_text)
                except ValueError:
                    continue
                rows.append({
                    'subject_id': subject_id,
                    'diagnosis': diagnosis,
                    'roi_name': roi_name,
                    'imaging_metric': 'roi_volume',
                    'value_numeric': value_numeric,
                    'source_pipeline': 'FreeSurfer',
                    'data_level': 'subject',
                    'statistic_type': 'raw_measure',
                    'hemisphere': None,
                })
    return rows

aparc_pattern = re.compile(r'^(\\S+)\\s+(\\d+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)\\s+(\\S+)')

def parse_aparc_stats(subject_id, diagnosis, hemi):
    aparc_path = FS_DIR / subject_id / 'stats' / f'{hemi}.aparc.stats'
    if not aparc_path.exists():
        return []
    rows = []
    for line in aparc_path.read_text(errors='ignore').splitlines():
        if not line or line.startswith('#'):
            continue
        match = aparc_pattern.match(line.strip())
        if not match:
            continue
        roi_name = match.group(1)
        gray_vol = match.group(4)
        thickness = match.group(6)
        for metric_name, metric_value in [('roi_volume', gray_vol), ('cortical_thickness', thickness)]:
            try:
                value_numeric = float(metric_value)
            except ValueError:
                continue
            rows.append({
                'subject_id': subject_id,
                'diagnosis': diagnosis,
                'roi_name': roi_name,
                'imaging_metric': metric_name,
                'value_numeric': value_numeric,
                'source_pipeline': 'FreeSurfer',
                'data_level': 'subject',
                'statistic_type': 'raw_measure',
                'hemisphere': hemi,
            })
    return rows

all_rows = []
for item in PILOT_SUBJECTS:
    sid = item['subject_id']
    dx = item['diagnosis']
    all_rows.extend(parse_aseg_stats(sid, dx))
    all_rows.extend(parse_aparc_stats(sid, dx, 'lh'))
    all_rows.extend(parse_aparc_stats(sid, dx, 'rh'))

structural_df = pd.DataFrame(all_rows)
structural_csv_path = OUTPUT_DIR / 'pilot_freesurfer_structural_features.csv'
if len(structural_df) > 0:
    structural_df.to_csv(structural_csv_path, index=False)

display(pd.DataFrame([
    {'item': 'n_rows', 'value': len(structural_df)},
    {'item': 'csv_exists', 'value': structural_csv_path.exists()},
    {'item': 'csv_path', 'value': str(structural_csv_path)},
]))
if len(structural_df) > 0:
    display(structural_df.head(20))


## Stage 6. Next handoff into the evidence-informed notebook

After the CSV exists, use this path in the later notebook:

/home/jovyan/outputs/pilot_freesurfer_structural_features.csv
